# Regridding: ERA5-Land yearly precipitation on the RhiresY grid

This example illustrates {func}`~grid_process.grid_interp.interp2grid` with three methods:

- `nearest`: value of the nearest source grid point;
- `idw`: inverse distance weighting of the `k` nearest grid points;
- `linear`: bilinear interpolation, since the source grid is regular.

The 2024 precipitation total of the ERA5-Land reanalysis (regular 0.1° grid, about 9 km) is regridded
to the 1 km grid of RhiresY, the MeteoSwiss yearly precipitation analysis based on rain gauges, in the
Swiss LV95 coordinate system. RhiresY 2024 is shown alongside as a reference.

Running this notebook needs a free account on the
[Copernicus Climate Data Store](https://cds.climate.copernicus.eu): your API token in `~/.cdsapirc`
and the licence of the ERA5-Land dataset accepted once on its download page.

*Data: contains modified Copernicus Climate Change Service information 2026, ERA5-Land monthly averaged
data (DOI: 10.24381/cds.68d2bb30), CC BY 4.0. MeteoSwiss open data, CC BY 4.0, source: MeteoSwiss.
Country borders: Natural Earth.*

In [ ]:
import time
import timeit

import pyproj  # noqa: F401  # import before earthkit.data to avoid a crash at exit (earthkit-data#1202)
import cartopy.crs as ccrs
import earthkit.data as ekd
import earthkit.plots as ekp
import numpy as np
import pandas as pd
import pooch
import xarray as xr

import grid_process.grid_interp as gri

METHODS = {
    "nearest": {"method": "nearest"},
    "idw": {"method": "idw", "k": 4},
    "linear": {"method": "linear"},
}
YEAR = 2024

## Destination grid and reference

The RhiresY file is a CF-compliant NetCDF file: `grid_process` finds the `E`/`N` coordinates from their
`standard_name` and the LV95 projection from the grid mapping variable, so the dataset can be used as
the destination grid as is. RhiresY has no value outside its analysis domain (Switzerland and some
border areas).

In [ ]:
rhires_path = pooch.retrieve(
    "https://data.geo.admin.ch/ch.meteoschweiz.ogd-surface-derived-grid/archive-ch/"
    "ogd-surface-derived-grid-archive.rhiresy_ch01r.swiss.lv95_20240101000000_20240101000000.nc",
    known_hash="sha256:7e896e6029826d2384bba0b184484fa8022d780af45b11b3cd07604dd1ae7e9c",
)
# time units "years since ..." cannot be decoded by xarray and are not needed here
dst_grid = xr.open_dataset(rhires_path, decode_times=False).squeeze("time", drop=True)
rhires = dst_grid["RhiresY"]
dst_grid

## Source data

ERA5-Land monthly means are retrieved from the Climate Data Store for a box around Switzerland with a
wide margin (`area` = North, West, South, East). Monthly means of accumulated variables are daily
totals in metres, so the yearly total is the sum over the months of the daily total times the number
of days in the month.

In [ ]:
request = {
    "product_type": ["monthly_averaged_reanalysis"],
    "variable": ["total_precipitation"],
    "year": [str(YEAR)],
    "month": [f"{month:02d}" for month in range(1, 13)],
    "time": ["00:00"],
    "area": [49.0, 4.5, 44.5, 12.0],
    "data_format": "netcdf",
    "download_format": "unarchived",
}
era5 = ekd.from_source("cds", "reanalysis-era5-land-monthly-means", request).to_xarray()

tp = era5["tp"]
days = tp["valid_time"].dt.days_in_month
ds = (1000 * (tp * days).sum("valid_time", skipna=False)).to_dataset(name="precipitation")
ds["precipitation"].attrs = {"long_name": f"precipitation total {YEAR}", "units": "mm"}
ds = ds.reset_coords(drop=True)
ds

## Interpolation

Each method is called once, and the duration of this first call is kept for the benchmark below. The
outputs are masked outside the RhiresY analysis domain.

In [ ]:
results, first_call = {}, {}
for label, kwargs in METHODS.items():
    t0 = time.perf_counter()
    out = gri.interp2grid(ds, dst_grid, **kwargs)
    first_call[label] = time.perf_counter() - t0
    results[label] = out["precipitation"].where(rhires.notnull())
results["linear"]

## Visual comparison

In [ ]:
lv95 = ccrs.epsg(2056)
style = ekp.styles.Style(
    levels=[400, 600, 800, 1000, 1200, 1400, 1600, 1800, 2000, 2500, 3000, 3500],
    colors="YlGnBu",
    extend="both",
    units="mm",
)
panels = {f"ERA5-Land, method={label}": out for label, out in results.items()} | {"RhiresY (reference)": rhires}

fig = ekp.Figure(rows=2, columns=2, figsize=(12, 7.5), crs=lv95)
for i, (title, field) in enumerate(panels.items()):
    subplot = fig.add_map(i // 2, i % 2)
    subplot.pcolormesh(field.values, x=field.E.values, y=field.N.values, style=style, transform=lv95)
    subplot.title(title)
fig.borders()
fig.legend(label=f"precipitation total {YEAR} [mm]")
fig.title(f"ERA5-Land precipitation {YEAR} regridded to the RhiresY 1 km grid")
fig.show()

## Performance benchmark

The first call includes building the neighbour search structures and reprojecting the coordinates,
which `grid_process` caches. Subsequent calls on the same grids (e.g. for other variables or years)
reuse them: their runtime is the mean ± standard deviation over 5 repetitions.

In [ ]:
def benchmark(func, data, target, methods: dict, first_call: dict, repeat: int = 5) -> pd.DataFrame:
    rows = []
    for label, kwargs in methods.items():
        warm = np.array(timeit.repeat(lambda: func(data, target, **kwargs), number=1, repeat=repeat))
        rows.append(
            {
                "method": label,
                "kwargs": ", ".join(f"{k}={v}" for k, v in kwargs.items() if k != "method"),
                "first call [s]": first_call[label],
                "subsequent calls [s]": f"{warm.mean():.3f} ± {warm.std():.3f}",
            }
        )
    return pd.DataFrame(rows).set_index("method").round(3)


print(f"{ds.latitude.size} × {ds.longitude.size} source grid points -> {dst_grid.N.size} × {dst_grid.E.size} cells")
benchmark(gri.interp2grid, ds, dst_grid, METHODS, first_call)

## Takeaways

TAKEAWAYS